# Copilot Audit Log — Direct Ingester (Fabric)

End-to-end ingestion notebook that **calls Microsoft Graph directly** from inside Fabric, parses the audit-log records on the fly, and writes a Delta table the **AI-in-One Dashboard** and **ValueLens** PBITs consume directly.

**No PowerShell, no intermediate CSVs, no separate parser.** Replaces the previous flow:

```
OLD:  PowerShell script → local CSV → manual move into Files/audit_raw/ → Copilot_Audit_Log_Parser.ipynb → Delta
NEW:  This notebook (Graph → parse → Delta)
```

**Output**: Lakehouse Delta table `Copilot_Interactions_Parsed` (same name + schema as the existing parser, so the PBIT works without changes).

**Schedule**: weekly (matches Microsoft Graph's 7-day audit-log query window). Use the **Schedule** button at the top of the notebook.

**Permissions**: app registration with `AuditLogsQuery.Read.All` (Application permission, admin-consented). No `Sites.Selected` needed — this path doesn't touch SharePoint.


## 1. Configuration

Fill in the four config values below. The client secret should ideally come from a Key Vault — see the commented alternative.

**Lakehouse Schemas note**: the default `OUTPUT_TABLE` is `dbo.Copilot_Interactions_Parsed`. This works for both schema-enabled Lakehouses (the new Fabric default) and non-schema Lakehouses — `dbo` is the implicit default schema either way. If your Lakehouse uses a different schema, change the prefix.


In [ ]:
# === CONFIG ===
TENANT_ID     = '<your-tenant-guid>'                  # Entra -> Overview -> Tenant ID
CLIENT_ID     = '<your-app-reg-client-id>'            # Entra -> App registrations -> your app -> Application (client) ID
CLIENT_SECRET = '<your-client-secret-value>'          # The secret VALUE (not Secret ID). Prefer Key Vault - see below.

# --- Load mode ---------------------------------------------------------------------
# 'backfill'    = pull BACKFILL_DAYS of history once (overwrite). Use for the initial load.
# 'incremental' = requery the trailing LOOKBACK_DAYS (for late arrivals) and merge by stable Id.
MODE          = 'incremental'
BACKFILL_DAYS = 180                                   # history to pull when MODE='backfill'
LOOKBACK_DAYS = 7                                     # trailing requery window + empty-table fallback
OUTPUT_TABLE  = 'dbo.Copilot_Interactions_Parsed'     # Delta table consumed by the PBIT (keep name/schema stable)

# --- Backfill scale tuning ---------------------------------------------------------
CHUNK_HOURS             = 8     # window size; backfill = many windows. 6-8h gives finer parallelism.
MAX_CONCURRENT_QUERIES  = 6     # audit query API caps ~5-10 concurrent + 429-throttles; keep bounded.
MAX_WAIT_MIN_PER_QUERY  = 240   # ceiling PER window (not the whole run). Resumable across reruns.
POLL_INTERVAL_SEC       = 30    # poll cadence per window
STAGING_DIR             = 'Files/_audit_staging'  # per-window JSONL + manifest; active runs only read their own windows.

# --- Production: load secret from Key Vault instead of hardcoding -------------------
# from notebookutils.credentials import getSecret
# CLIENT_SECRET = getSecret('https://<your-vault>.vault.azure.net/', 'CopilotAuditAppSecret')


## 2. Authenticate to Microsoft Graph

Service principal (client secret) flow — same auth pattern as the PowerShell scripts.


In [ ]:
import requests, time
from requests.adapters import HTTPAdapter
try:
    from urllib3.util.retry import Retry
except Exception:
    from requests.packages.urllib3.util.retry import Retry

# Resilient HTTP session. The Microsoft Graph security/auditLog endpoints intermittently return
# 502/503/504 (gateway timeouts) and 429 (throttling), especially during the poll + records-fetch
# phases on busy tenants. A shared Session with urllib3 Retry transparently retries those with
# exponential backoff + Retry-After, so a transient blip no longer aborts the whole run.
def _build_session() -> requests.Session:
    s = requests.Session()
    retry = Retry(
        total=8,                       # up to 8 attempts per request
        connect=5, read=5, status=8,
        backoff_factor=2,              # 0,2,4,8,16,32... seconds (capped by urllib3)
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=frozenset(['GET', 'POST']),
        respect_retry_after_header=True,
        raise_on_status=False,
    )
    adapter = HTTPAdapter(max_retries=retry, pool_connections=8, pool_maxsize=8)
    s.mount('https://', adapter)
    s.mount('http://', adapter)
    return s

SESSION = _build_session()

# Belt-and-braces wrapper: urllib3 Retry handles most cases, but we add an outer retry loop so a
# request that *still* surfaces a transient status (or a connection error) after the adapter gives
# up is retried a few more times rather than killing the notebook.
_TRANSIENT = {429, 500, 502, 503, 504}

def _request(method: str, url: str, *, attempts: int = 5, **kwargs):
    kwargs.setdefault('timeout', 60)
    last = None
    for i in range(1, attempts + 1):
        try:
            r = SESSION.request(method, url, **kwargs)
        except requests.exceptions.RequestException as e:
            last = e
            if i == attempts:
                raise
            wait = min(60, 2 ** i)
            print(f'    transient network error ({type(e).__name__}); retry {i}/{attempts} in {wait}s')
            time.sleep(wait)
            continue
        if r.status_code in _TRANSIENT and i < attempts:
            wait = min(60, 2 ** i)
            print(f'    transient HTTP {r.status_code} on {method} {url.split("?")[0].rsplit("/",1)[-1]}; '
                  f'retry {i}/{attempts} in {wait}s')
            time.sleep(wait)
            continue
        return r
    return r  # pragma: no cover

# Auto-refreshing app-only Graph token. Long (multi-hour) chunked runs would otherwise
# fail when the 1-hour token expires mid-pull, so we transparently refresh every ~50 min.
def _get_graph_token() -> str:
    url  = f'https://login.microsoftonline.com/{TENANT_ID}/oauth2/v2.0/token'
    body = {
        'client_id':     CLIENT_ID,
        'scope':         'https://graph.microsoft.com/.default',
        'client_secret': CLIENT_SECRET,
        'grant_type':    'client_credentials',
    }
    r = _request('POST', url, data=body, timeout=30)
    r.raise_for_status()
    return r.json()['access_token']

_token = {'value': None, 'ts': 0.0}

def get_headers() -> dict:
    if _token['value'] is None or (time.time() - _token['ts']) > 50 * 60:
        _token['value'] = _get_graph_token()
        _token['ts'] = time.time()
    return {'Authorization': f'Bearer {_token["value"]}', 'Content-Type': 'application/json'}

get_headers()
print('\u2713 Graph token acquired (auto-refreshing, retry-resilient session).')


## 3. Build query windows + the `create_query` helper

The lookback is split into smaller time windows (`CHUNK_HOURS`). Each window is its own async
audit query, which completes far faster than one giant query — the main cause of timeouts on
large tenants — and lets the run checkpoint window-by-window.

In [ ]:
from datetime import datetime, timezone, timedelta
import hashlib, json

WINDOW_KEY_VERSION = 2
KEY_COLUMNS = ['Id', 'Source_RecordKey', 'Source_MessageKey', 'Source_ResourceKey']
_RESOURCE_KEY_FIELDS = ('Type', 'Action', 'SiteUrl', 'SensitivityLabelId')
_SYNTHETIC_AUDIT_FIELDS = frozenset({'_StableKey', '_StableOrdinal'})


def _as_utc_datetime(value):
    if value in (None, ''):
        return None
    if isinstance(value, datetime):
        dt = value
    else:
        text = str(value).strip()
        if not text:
            return None
        if text.endswith('Z'):
            text = text[:-1] + '+00:00'
        dt = datetime.fromisoformat(text)
    if dt.tzinfo is None:
        return dt.replace(tzinfo=timezone.utc)
    return dt.astimezone(timezone.utc)


def _norm_text(value) -> str:
    if value is None:
        return ''
    return ' '.join(str(value).strip().split())


def _coerce_json_value(value):
    if value is None:
        return None
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return None
        try:
            return json.loads(text)
        except json.JSONDecodeError:
            return text
    return value


def _has_meaningful_value(value) -> bool:
    if isinstance(value, dict):
        return any(_has_meaningful_value(v) for k, v in value.items() if k not in _SYNTHETIC_AUDIT_FIELDS)
    if isinstance(value, list):
        return any(_has_meaningful_value(v) for v in value)
    if value is None:
        return False
    if isinstance(value, str):
        return _norm_text(value) != ''
    return True


def _canonical_json_text(value) -> str:
    return json.dumps(value, sort_keys=True, ensure_ascii=False, separators=(',', ':'))


def _stable_message_base_key(message) -> str:
    message = dict(message or {})
    msg_id = _norm_text(message.get('Id'))
    if msg_id:
        return f'mid:{msg_id}'
    payload = {k: v for k, v in message.items() if k not in _SYNTHETIC_AUDIT_FIELDS and k != 'Id'}
    return 'msgf:' + hashlib.sha256(_canonical_json_text(payload).encode('utf-8')).hexdigest()


def _stable_resource_base_key(resource) -> str:
    resource = dict(resource or {})
    payload = {k: v for k, v in resource.items() if k not in _SYNTHETIC_AUDIT_FIELDS}
    if not _has_meaningful_value(payload):
        return 'resource:none'
    return 'resource:' + hashlib.sha256(_canonical_json_text(payload).encode('utf-8')).hexdigest()


def _stable_sequence(items, base_key_builder, path) -> list:
    prepared = []
    for item in items or []:
        canonical_item = _canonicalize_json(item, path)
        if not isinstance(canonical_item, dict):
            canonical_item = {'Value': canonical_item}
        canonical_text = _canonical_json_text(canonical_item)
        base_key = base_key_builder(canonical_item)
        prepared.append((base_key, canonical_text, canonical_item))
    prepared.sort(key=lambda entry: (entry[0], entry[1]))
    totals = {}
    for base_key, _canonical_text, _item in prepared:
        totals[base_key] = totals.get(base_key, 0) + 1
    seen = {}
    stable = []
    for ordinal, (base_key, _canonical_text, canonical_item) in enumerate(prepared):
        seen[base_key] = seen.get(base_key, 0) + 1
        stable_key = base_key if totals[base_key] == 1 else f'{base_key}#{seen[base_key]}'
        enriched = dict(canonical_item)
        enriched['_StableKey'] = stable_key
        enriched['_StableOrdinal'] = ordinal
        stable.append(enriched)
    return stable


def _canonicalize_json(value, path=()):
    value = _coerce_json_value(value) if not path else value
    if isinstance(value, dict):
        return {
            str(key): _canonicalize_json(val, path + (str(key),))
            for key, val in sorted(value.items(), key=lambda item: str(item[0]))
            if key not in _SYNTHETIC_AUDIT_FIELDS
        }
    if isinstance(value, list):
        if path == ('CopilotEventData', 'Messages'):
            return _stable_sequence(value, _stable_message_base_key, path)
        if path == ('CopilotEventData', 'AccessedResources'):
            return _stable_sequence(value, _stable_resource_base_key, path)
        return [_canonicalize_json(item, path) for item in value]
    return value


def canonicalize_audit_payload(audit_data):
    value = _coerce_json_value(audit_data)
    if value is None:
        return None
    return _canonicalize_json(value)


def determine_incremental_start(end_dt, high_water_mark, lookback_days: int):
    lookback_days = max(int(lookback_days), 0)
    lookback_floor = end_dt - timedelta(days=lookback_days)
    hw = _as_utc_datetime(high_water_mark)
    if hw is None:
        return lookback_floor
    return min(hw, lookback_floor)


def validate_incremental_target_columns(columns, required_columns, table_name: str):
    cols = set(columns or [])
    missing = [c for c in required_columns if c not in cols]
    if missing:
        raise RuntimeError(
            f"Legacy table {table_name} is missing stable key columns {missing}. "
            "Run one MODE='backfill' overwrite to upgrade the parsed audit table before incremental resumes."
        )


def build_source_record_key(
    record_id,
    creation_date=None,
    operation=None,
    audit_data=None,
    record_type=None,
    associated_admin_units=None,
    associated_admin_units_names=None,
) -> str:
    rid = _norm_text(record_id)
    if rid:
        return f"rid:{rid}"
    payload = {
        'CreationDate': _norm_text(creation_date),
        'Operation': _norm_text(operation),
        'RecordType': _norm_text(record_type),
        'AuditData': canonicalize_audit_payload(audit_data),
        'AssociatedAdminUnits': _canonicalize_json(associated_admin_units or []),
        'AssociatedAdminUnitsNames': _canonicalize_json(associated_admin_units_names or []),
    }
    return 'synthetic:' + hashlib.sha256(_canonical_json_text(payload).encode('utf-8')).hexdigest()


def canonicalize_audit_record(record) -> dict:
    record = record or {}
    audit_payload = canonicalize_audit_payload(record.get('auditData'))
    return {
        'RecordId': record.get('id'),
        'CreationDate': record.get('createdDateTime'),
        'RecordType': record.get('auditLogRecordType'),
        'Operation': record.get('operation'),
        'AuditData': _canonical_json_text(audit_payload) if audit_payload is not None else None,
        'SourceRecordKey': build_source_record_key(
            record.get('id'),
            record.get('createdDateTime'),
            record.get('operation'),
            audit_payload,
            record_type=record.get('auditLogRecordType'),
            associated_admin_units=record.get('associatedAdminUnits', []),
            associated_admin_units_names=record.get('associatedAdminUnitsNames', []),
        ),
        'AssociatedAdminUnits': _canonical_json_text(_canonicalize_json(record.get('associatedAdminUnits', []))),
        'AssociatedAdminUnitsNames': _canonical_json_text(_canonicalize_json(record.get('associatedAdminUnitsNames', []))),
    }


def stable_window_key(win_start, win_end) -> str:
    ws = _as_utc_datetime(win_start)
    we = _as_utc_datetime(win_end)
    return f"v{WINDOW_KEY_VERSION}_{ws:%Y%m%d%H%M}_{we:%Y%m%d%H%M}"


if MODE not in ('backfill', 'incremental'):
    raise ValueError(f"Unsupported MODE={MODE!r}; expected 'backfill' or 'incremental'.")

end_date = datetime.now(timezone.utc)
if MODE == 'backfill':
    start_date = end_date - timedelta(days=BACKFILL_DAYS)
    WRITE_MODE = 'overwrite'
else:
    WRITE_MODE = 'merge'
    _hw = None
    if spark.catalog.tableExists(OUTPUT_TABLE):
        from pyspark.sql import functions as F
        _existing = spark.table(OUTPUT_TABLE)
        validate_incremental_target_columns(_existing.columns, KEY_COLUMNS, OUTPUT_TABLE)
        _hw = _existing.agg(F.max('CreationDate').alias('m')).collect()[0]['m']
    start_date = determine_incremental_start(end_date, _hw, LOOKBACK_DAYS)
    if _hw:
        print(
            f'incremental: high-water mark = {_as_utc_datetime(_hw):%Y-%m-%d %H:%M}; '
            f'requerying trailing {LOOKBACK_DAYS}d from {start_date:%Y-%m-%d %H:%M}'
        )
    else:
        print(f'incremental: no prior data, defaulting to last {LOOKBACK_DAYS}d')

_anchor_hour = start_date.hour - (start_date.hour % CHUNK_HOURS)
_cur = start_date.replace(hour=_anchor_hour, minute=0, second=0, microsecond=0)
windows = []
while _cur < end_date:
    _nxt = min(_cur + timedelta(hours=CHUNK_HOURS), end_date)
    windows.append((_cur, _nxt))
    _cur = _nxt
print(f'MODE={MODE}: {start_date:%Y-%m-%d} -> {end_date:%Y-%m-%d}  =>  {len(windows)} window(s) of {CHUNK_HOURS}h, WRITE_MODE={WRITE_MODE}')


def create_query(win_start, win_end) -> str:
    body = {
        'displayName':         f'Copilot Interactions {win_start:%Y%m%d%H%M}-{win_end:%Y%m%d%H%M}',
        'filterStartDateTime': win_start.isoformat(),
        'filterEndDateTime':   win_end.isoformat(),
        'recordTypeFilters':   ['copilotInteraction'],
        'operationFilters':    [],
    }
    r = _request('POST', 'https://graph.microsoft.com/beta/security/auditLog/queries',
                 json=body, headers=get_headers(), timeout=30)
    r.raise_for_status()
    return r.json()['id']


## 4. Resumable wait helper

Polls an async query until it succeeds, with a **generous, configurable** ceiling
(`MAX_WAIT_MIN_PER_QUERY`) instead of a hard 30-minute cut-off. The token auto-refreshes while
waiting, so multi-hour jobs don't fail on expiry.

In [ ]:
import time


def _read_query_status(query_id: str):
    r = _request(
        'GET',
        f'https://graph.microsoft.com/beta/security/auditLog/queries/{query_id}',
        headers=get_headers(),
        timeout=30,
    )
    r.raise_for_status()
    payload = r.json()
    if not isinstance(payload, dict):
        raise RuntimeError(
            f'Query {query_id} returned malformed status payload type: {type(payload).__name__}'
        )
    status = payload.get('status')
    if not isinstance(status, str) or not status.strip():
        raise RuntimeError(f'Query {query_id} returned missing/invalid status: {status!r}')
    return payload, status.strip().lower()


def wait_for_query(query_id: str, max_wait_min: int = MAX_WAIT_MIN_PER_QUERY):
    started = time.time()
    deadline = started + max_wait_min * 60
    polls = 0
    while time.time() < deadline:
        polls += 1
        payload, status = _read_query_status(query_id)
        if status == 'succeeded':
            return payload
        if status in ('failed', 'cancelled'):
            raise RuntimeError(f'Query {query_id} ended with status: {status}')
        if status not in ('notstarted', 'running'):
            raise RuntimeError(f'Query {query_id} returned unexpected status: {status}')
        if polls % 4 == 0:
            mins = int((time.time() - started) // 60)
            print(f'    … still {status} ({mins}m elapsed, {polls} polls)')
        time.sleep(POLL_INTERVAL_SEC)
    raise TimeoutError(
        f'Query {query_id} did not succeed within {max_wait_min} min. '
        f'Increase MAX_WAIT_MIN_PER_QUERY or reduce CHUNK_HOURS.'
    )


## 5. Fetch records → stream to Lakehouse Files (chunked, bounded memory)

Each page of results is written straight to Lakehouse **Files** as JSON-lines, so the full
result set is **never** held in driver memory — the other main cause of OOM/timeouts on large
tenants. Set `WRITE_MODE = 'overwrite'` to clear the staging area first.

In [ ]:
import glob, os, json, threading
from urllib.parse import urlparse
from concurrent.futures import ThreadPoolExecutor, as_completed

# Resumable, bounded-concurrency backfill. Each window's records stream to a JSONL file and its
# status is recorded in a manifest; rerun skips older succeeded windows, but recent LOOKBACK_DAYS
# windows are intentionally re-queried for late arrivals. Active runs only read the current window set.
STAGING_ABS = '/lakehouse/default/' + STAGING_DIR.rstrip('/')
MANIFEST = os.path.join(STAGING_ABS, '_manifest.json')
MANIFEST_LOCK = threading.RLock()
os.makedirs(STAGING_ABS, exist_ok=True)


def list_window_files(base_dir: str, key: str, include_partial: bool = False):
    patterns = [os.path.join(base_dir, f'win_{key}_*.jsonl')]
    if include_partial:
        patterns.append(os.path.join(base_dir, f'win_{key}_*.jsonl.partial'))
    files = []
    for pattern in patterns:
        files.extend(glob.glob(pattern))
    return sorted(set(files))


def purge_window_files(base_dir: str, key: str, include_partial: bool = True) -> int:
    removed = 0
    for path in list_window_files(base_dir, key, include_partial=include_partial):
        if os.path.isfile(path):
            os.remove(path)
            removed += 1
    return removed


def list_active_stage_files(base_dir: str, active_keys) -> list:
    files = []
    for key in sorted(set(active_keys or [])):
        files.extend(list_window_files(base_dir, key, include_partial=False))
    return sorted(files)


def _allowed_manifest_statuses():
    return {'querying', 'waiting', 'draining', 'succeeded', 'failed'}


def _validate_manifest_payload(data):
    if not isinstance(data, dict):
        raise RuntimeError(
            f'Manifest {MANIFEST} must be a JSON object keyed by window id; found {type(data).__name__}.'
        )
    allowed = _allowed_manifest_statuses()
    for key, entry in data.items():
        if not isinstance(key, str) or not key.strip():
            raise RuntimeError(f'Manifest {MANIFEST} has an invalid window key: {key!r}.')
        if not isinstance(entry, dict):
            raise RuntimeError(
                f'Manifest {MANIFEST} entry {key!r} must be a JSON object; found {type(entry).__name__}.'
            )
        status = entry.get('status')
        if not isinstance(status, str) or status not in allowed:
            raise RuntimeError(
                f"Manifest {MANIFEST} entry {key!r} has invalid status {status!r}; "
                f'expected one of {sorted(allowed)}.'
            )
    return data


def _load_manifest():
    try:
        with open(MANIFEST, encoding='utf-8') as f:
            data = json.load(f)
    except FileNotFoundError:
        return {}
    except json.JSONDecodeError as e:
        raise RuntimeError(f'Manifest {MANIFEST} is not valid JSON: {e}') from e
    except UnicodeDecodeError as e:
        raise RuntimeError(f'Manifest {MANIFEST} is not valid UTF-8 JSON: {e}') from e
    return _validate_manifest_payload(data)


def _manifest_temp_path():
    base = os.path.basename(MANIFEST)
    return os.path.join(STAGING_ABS, f'.{base}.{os.getpid()}.{threading.get_ident()}.tmp')


def _save_manifest(m):
    payload = _validate_manifest_payload(dict(m))
    os.makedirs(STAGING_ABS, exist_ok=True)
    temp_path = _manifest_temp_path()
    try:
        with open(temp_path, 'w', encoding='utf-8') as f:
            json.dump(payload, f, indent=2, sort_keys=True)
            f.flush()
            os.fsync(f.fileno())
        os.replace(temp_path, MANIFEST)
    finally:
        if os.path.exists(temp_path):
            os.remove(temp_path)


def manifest_succeeded(entry) -> bool:
    return isinstance(entry, dict) and entry.get('status') == 'succeeded'


def should_refresh_window(win_end, manifest_entry, trailing_cutoff) -> bool:
    if not manifest_succeeded(manifest_entry):
        return True
    completed_at = _as_utc_datetime((manifest_entry or {}).get('completed_at'))
    return _as_utc_datetime(win_end) > _as_utc_datetime(trailing_cutoff) or completed_at is None


def _read_manifest_entry(key: str) -> dict:
    with MANIFEST_LOCK:
        current = _load_manifest()
        manifest.clear()
        manifest.update(current)
        return dict(current.get(key, {}))


def _mark_window(key: str, status: str, **extra):
    if not isinstance(key, str) or not key.strip():
        raise ValueError(f'Window key must be a non-empty string; found {key!r}.')
    if not isinstance(status, str) or status not in _allowed_manifest_statuses():
        raise ValueError(f'Unsupported manifest status {status!r}.')
    with MANIFEST_LOCK:
        current = _load_manifest()
        entry = dict(current.get(key, {}))
        entry.update(extra)
        entry['status'] = status
        current[key] = entry
        _save_manifest(current)
        manifest.clear()
        manifest.update(current)


def _row(r):
    return canonicalize_audit_record(r)


def _is_graph_records_url(url: str, qid: str) -> bool:
    parsed = urlparse(url)
    return (
        parsed.scheme == 'https'
        and parsed.netloc == 'graph.microsoft.com'
        and parsed.path.startswith(f'/beta/security/auditLog/queries/{qid}/records')
    )


def _validate_graph_records_url(url, qid: str) -> str:
    if not isinstance(url, str) or not url.strip():
        raise RuntimeError(f'Query {qid} returned missing/invalid @odata.nextLink: {url!r}')
    url = url.strip()
    if not _is_graph_records_url(url, qid):
        raise RuntimeError(f'Query {qid} returned non-Graph records continuation URL: {url}')
    return url


def _validate_records_page(qid: str, current_url: str, payload):
    if not isinstance(payload, dict):
        raise RuntimeError(
            f'Query {qid} returned malformed records payload type {type(payload).__name__} for {current_url}'
        )
    if 'value' not in payload:
        raise RuntimeError(f'Query {qid} records payload missing value list for {current_url}')
    vals = payload['value']
    if not isinstance(vals, list):
        raise RuntimeError(f'Query {qid} records payload value must be a list for {current_url}')
    for index, record in enumerate(vals):
        if not isinstance(record, dict):
            raise RuntimeError(
                f'Query {qid} records payload item {index} must be an object for {current_url}'
            )
    next_link = payload.get('@odata.nextLink')
    if next_link is None:
        return vals, None
    return vals, _validate_graph_records_url(next_link, qid)


def _drain(qid, key):
    n, page = 0, 0
    partial_files = []
    purge_window_files(STAGING_ABS, key, include_partial=True)
    url = _validate_graph_records_url(
        f'https://graph.microsoft.com/beta/security/auditLog/queries/{qid}/records?$top=999',
        qid,
    )
    seen_urls = set()
    while url:
        if url in seen_urls:
            raise RuntimeError(f'Query {qid} returned repeated pagination link cycle: {url}')
        seen_urls.add(url)
        r = _request('GET', url, headers=get_headers(), timeout=60)
        r.raise_for_status()
        vals, next_url = _validate_records_page(qid, url, r.json())
        if next_url in seen_urls:
            raise RuntimeError(f'Query {qid} returned repeated pagination link cycle: {next_url}')
        if vals:
            part_path = os.path.join(STAGING_ABS, f'win_{key}_{page:04d}.jsonl.partial')
            with open(part_path, 'w', encoding='utf-8') as f:
                for rec in vals:
                    f.write(json.dumps(_row(rec)) + '\n')
            partial_files.append(part_path)
            page += 1
            n += len(vals)
        url = next_url
    for part_path in partial_files:
        os.replace(part_path, part_path[:-8])
    return n, page


def _process(win):
    ws, we = win
    key = stable_window_key(ws, we)
    entry = _read_manifest_entry(key)
    trailing_cutoff = end_date - timedelta(days=max(int(LOOKBACK_DAYS), 0))
    if not should_refresh_window(we, entry, trailing_cutoff):
        return key, entry.get('rows', 0), 'skip'
    _mark_window(
        key,
        'querying',
        window_start=_as_utc_datetime(ws).isoformat(),
        window_end=_as_utc_datetime(we).isoformat(),
        refreshed_at=datetime.now(timezone.utc).isoformat(),
    )
    qid = None
    try:
        qid = create_query(ws, we)
        _mark_window(key, 'waiting', query_id=qid)
        wait_for_query(qid)
        _mark_window(key, 'draining', query_id=qid)
        n, pages = _drain(qid, key)
        _mark_window(
            key,
            'succeeded',
            query_id=qid,
            rows=n,
            pages=pages,
            files=[os.path.basename(p) for p in list_window_files(STAGING_ABS, key)],
            completed_at=datetime.now(timezone.utc).isoformat(),
        )
        return key, n, 'done'
    except Exception as e:
        purge_window_files(STAGING_ABS, key, include_partial=True)
        failure = {
            'error': f'{type(e).__name__}: {e}',
            'completed_at': None,
        }
        if qid:
            failure['query_id'] = qid
        _mark_window(key, 'failed', **failure)
        raise


with MANIFEST_LOCK:
    manifest = _load_manifest()
ACTIVE_WINDOW_KEYS = {stable_window_key(ws, we) for ws, we in windows}
trailing_cutoff = end_date - timedelta(days=max(int(LOOKBACK_DAYS), 0))
pending = [w for w in windows if should_refresh_window(w[1], manifest.get(stable_window_key(*w)), trailing_cutoff)]
print(f'{len(windows)-len(pending)} window(s) already reusable; {len(pending)} to fetch (<= {MAX_CONCURRENT_QUERIES} at a time).')
total = sum((manifest.get(key, {}) or {}).get('rows', 0) for key in ACTIVE_WINDOW_KEYS)
done = 0
with ThreadPoolExecutor(max_workers=MAX_CONCURRENT_QUERIES) as ex:
    futs = {ex.submit(_process, w): w for w in pending}
    for fut in as_completed(futs):
        key, n, how = fut.result()
        total += n if how == 'done' else 0
        done += 1
        print(f'  [{done}/{len(pending)}] {key} {how} (+{n:,})')
print(f'Prepared {len(ACTIVE_WINDOW_KEYS)} active window(s); staged files will be read only from this run scope. Manifest: {MANIFEST}')


## 6. Read the staged records (distributed)

Load the staged JSON-lines with Spark — a distributed read, not a driver-side
`createDataFrame` from a giant Python list.

In [ ]:
import os
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, ArrayType

_cols = ['RecordId', 'CreationDate', 'RecordType', 'Operation',
         'AuditData', 'SourceRecordKey', 'AssociatedAdminUnits', 'AssociatedAdminUnitsNames']

# Read only the per-window JSONL files that belong to the current window set. This prevents
# historical staging files from being re-appended on later incremental runs.
_files = list_active_stage_files(STAGING_ABS, ACTIVE_WINDOW_KEYS)
if not _files:
    _empty = StructType([StructField(c, StringType()) for c in _cols])
    raw = spark.createDataFrame([], _empty)
else:
    raw = spark.read.json(_files)

raw = raw.persist()
print(f'Raw DataFrame rows: {raw.count():,} (from {len(_files)} active window file(s))')


## 7. Parse `AuditData` JSON

Schema mirrors the existing `Copilot_Audit_Log_Parser.ipynb` — only the fields the PBIT consumes.


In [ ]:
audit_schema = StructType([
    StructField('CreationTime', StringType()),
    StructField('UserId', StringType()),
    StructField('Workload', StringType()),
    StructField('ApplicationName', StringType()),
    StructField('ClientRegion', StringType()),
    StructField('AgentId', StringType()),
    StructField('AgentName', StringType()),
    StructField('AppIdentity', StructType([
        StructField('AppId', StringType()),
        StructField('DisplayName', StringType()),
        StructField('PublisherId', StringType()),
    ])),
    StructField('CopilotEventData', StructType([
        StructField('AppHost', StringType()),
        StructField('ThreadId', StringType()),
        StructField('SensitivityLabelId', StringType()),
        StructField('Contexts', ArrayType(StructType([
            StructField('Type', StringType())
        ]))),
        StructField('AISystemPlugin', ArrayType(StructType([
            StructField('Id', StringType()),
            StructField('Name', StringType())
        ]))),
        StructField('ModelTransparencyDetails', ArrayType(StructType([
            StructField('ModelName', StringType())
        ]))),
        StructField('AccessedResources', ArrayType(StructType([
            StructField('Type', StringType()),
            StructField('Action', StringType()),
            StructField('SiteUrl', StringType()),
            StructField('SensitivityLabelId', StringType()),
            StructField('_StableKey', StringType()),
            StructField('_StableOrdinal', StringType()),
        ]))),
        StructField('Messages', ArrayType(StructType([
            StructField('Id', StringType()),
            StructField('isPrompt', StringType()),
            StructField('_StableKey', StringType()),
            StructField('_StableOrdinal', StringType()),
        ]))),
    ])),
])

parsed = (raw
    .filter(F.col('AuditData').isNotNull() & (F.length('AuditData') > 10))
    .withColumn('j', F.from_json('AuditData', audit_schema))
    .filter(F.col('j').isNotNull()))


## 8. Flatten + fan out per (prompt × resource)

Mirrors the M-query exactly: drop responses, expand resources. One row per prompt-resource pair.


In [ ]:
_empty_message = F.struct(
    F.lit(None).cast(StringType()).alias('Id'),
    F.lit(None).cast(StringType()).alias('isPrompt'),
    F.lit(None).cast(StringType()).alias('_StableKey'),
    F.lit(None).cast('long').alias('_StableOrdinal'),
)
_empty_resource = F.struct(
    F.lit(None).cast(StringType()).alias('Type'),
    F.lit(None).cast(StringType()).alias('Action'),
    F.lit(None).cast(StringType()).alias('SiteUrl'),
    F.lit(None).cast(StringType()).alias('SensitivityLabelId'),
    F.lit('resource:none').cast(StringType()).alias('_StableKey'),
    F.lit(0).cast('long').alias('_StableOrdinal'),
)


def _nonblank(col):
    return F.length(F.trim(F.coalesce(col.cast('string'), F.lit('')))) > 0


_base = (
    parsed
    .select(
        F.col('RecordId'),
        F.col('Operation'),
        F.col('AuditData').alias('_AuditDataRaw'),
        F.col('SourceRecordKey').alias('_SourceRecordKey'),
        F.col('j.CreationTime').cast('timestamp').alias('CreationDate'),
        F.col('j.AgentId').alias('AgentId'),
        F.col('j.AgentName').alias('AgentName'),
        F.col('j.AppIdentity.AppId').alias('AppIdentity_AppId'),
        F.col('j.AppIdentity.DisplayName').alias('AppIdentity_DisplayName'),
        F.col('j.AppIdentity.PublisherId').alias('AppIdentity_PublisherId'),
        F.col('j.ApplicationName').alias('ApplicationName'),
        F.col('j.ClientRegion').alias('ClientRegion'),
        F.col('j.UserId').alias('Audit_UserId'),
        F.lower(F.trim(F.col('j.UserId'))).alias('Audit_UserId_Normalized'),
        F.col('j.Workload').alias('Workload'),
        F.col('j.CopilotEventData.AppHost').alias('AppHost'),
        F.col('j.CopilotEventData.ThreadId').alias('ThreadId'),
        F.col('j.CopilotEventData.SensitivityLabelId').alias('SensitivityLabelId'),
        F.col('j.CopilotEventData.Contexts')[0]['Type'].alias('Context_Type'),
        F.col('j.CopilotEventData.AISystemPlugin')[0]['Id'].alias('AISystemPlugin_Id'),
        F.col('j.CopilotEventData.AISystemPlugin')[0]['Name'].alias('AISystemPlugin_Name'),
        F.col('j.CopilotEventData.ModelTransparencyDetails')[0]['ModelName']
            .alias('ModelTransparencyDetails_ModelName'),
        F.col('j.CopilotEventData.AccessedResources').alias('Resources'),
        F.col('j.CopilotEventData.Messages').alias('Messages'),
        F.size('j.CopilotEventData.AccessedResources').alias('Resource_Count'),
    )
)

_ambiguous_synthetic_records = (
    _base
    .filter(~_nonblank(F.col('RecordId')) & _nonblank(F.col('_SourceRecordKey')))
    .groupBy('_SourceRecordKey')
    .count()
    .filter(F.col('count') > 1)
    .limit(1)
    .collect()
)
if _ambiguous_synthetic_records:
    raise RuntimeError(
        "Missing RecordId produced ambiguous synthetic SourceRecordKey values; refusing unsafe dedup."
    )

flat = (
    _base
    .withColumn(
        'Source_RecordKey',
        F.when(_nonblank(F.col('_SourceRecordKey')), F.trim(F.col('_SourceRecordKey').cast('string')))
         .when(_nonblank(F.col('RecordId')), F.concat(F.lit('rid:'), F.trim(F.col('RecordId').cast('string'))))
    )
    .select(
        '*',
        F.posexplode_outer(
            F.when(F.size('Messages') > 0, F.col('Messages')).otherwise(F.array(_empty_message))
        ).alias('Message_ArrayOrdinal', 'msg'),
    )
    .filter(F.lower(F.col('msg.isPrompt').cast('string')) == 'true')
    .select(
        '*',
        F.posexplode_outer(
            F.when(F.size('Resources') > 0, F.col('Resources')).otherwise(F.array(_empty_resource))
        ).alias('Resource_ArrayOrdinal', 'res'),
    )
    .withColumn('Source_MessageKey', F.trim(F.col('msg._StableKey').cast('string')))
    .withColumn('Source_ResourceKey', F.trim(F.col('res._StableKey').cast('string')))
    .withColumn(
        'Message_Ordinal',
        F.coalesce(F.col('msg._StableOrdinal').cast('long'), F.col('Message_ArrayOrdinal').cast('long')),
    )
    .withColumn(
        'Resource_Ordinal',
        F.coalesce(F.col('res._StableOrdinal').cast('long'), F.col('Resource_ArrayOrdinal').cast('long'), F.lit(0).cast('long')),
    )
)

_missing_identity_keys = (
    flat
    .filter(
        ~_nonblank(F.col('Source_RecordKey'))
        | ~_nonblank(F.col('Source_MessageKey'))
        | ~_nonblank(F.col('Source_ResourceKey'))
    )
    .limit(1)
    .collect()
)
if _missing_identity_keys:
    raise RuntimeError(
        "Canonical audit identity fields are missing from staged data; rerun ingestion with the canonical stage helpers."
    )

flat = (
    flat
    .withColumn(
        'Id',
        F.sha2(
            F.concat_ws(
                '||',
                F.col('Source_RecordKey'),
                F.col('Source_MessageKey'),
                F.col('Source_ResourceKey'),
            ),
            256,
        ),
    )
    .select(
        'Id', 'RecordId', 'Source_RecordKey', 'Source_MessageKey', 'Source_ResourceKey',
        'CreationDate', 'AgentId', 'AgentName',
        'AppIdentity_AppId', 'AppIdentity_DisplayName', 'AppIdentity_PublisherId',
        'ApplicationName', 'ClientRegion',
        'Audit_UserId', 'Audit_UserId_Normalized', 'Workload',
        'AppHost', 'ThreadId', 'SensitivityLabelId',
        'Context_Type',
        'AISystemPlugin_Id', 'AISystemPlugin_Name',
        'ModelTransparencyDetails_ModelName',
        F.col('res.Type').alias('AccessedResource_Type'),
        F.col('res.Action').alias('AccessedResource_Action'),
        F.col('res.SiteUrl').alias('AccessedResource_SiteUrl'),
        F.col('res.SensitivityLabelId').alias('AccessedResource_SensitivityLabelId'),
        F.col('msg.Id').alias('Message_Id'),
        F.col('msg.isPrompt').alias('Message_isPrompt'),
        F.col('Message_Ordinal').cast('long').alias('Message_Ordinal'),
        F.col('Resource_Ordinal').cast('long').alias('Resource_Ordinal'),
        F.coalesce(F.col('Resource_Count'), F.lit(1)).cast('long').alias('Resource_Count'),
        F.to_date('CreationDate').alias('InteractionDate'),
        F.expr("date_trunc('week', CreationDate)").cast('date').alias('WeekStart'),
        F.expr("date_trunc('month', CreationDate)").cast('date').alias('MonthStart'),
    )
    .dropDuplicates(['Id'])
)


## 9. Derive `Agent_TitleID` + `Agent_EntraId`

`Agent_TitleID` keeps the legacy Copilot Studio logic (existing parser + M-query).
`Agent_EntraId` captures the **Microsoft Entra Agent ID** GUID that Agent 365 now
stamps into the audit `AgentId` (instead of the `CopilotStudio.Declarative.{title}`
string). Without it, Entra-registered agents fall through to NULL and drop out of
the Agents join. Both keys are emitted so old and new agents resolve via the
registry crosswalk; neither overwrites the other.


In [ ]:
flat = flat.withColumn('Agent_TitleID', F.expr(r'''
    CASE
      WHEN AgentId IS NULL THEN NULL
      WHEN AgentId LIKE '%CopilotStudio.Declarative.%' THEN
           split(split(AgentId, 'CopilotStudio\\.Declarative\\.')[1], '\\.')[0]
      WHEN AgentId LIKE 'P\\_%' OR AgentId LIKE 'T\\_%' THEN
           split(AgentId, '\\.')[0]
      ELSE NULL
    END
'''))

# Microsoft Entra Agent ID (Agent 365): the audit AgentId now carries an Entra
# Agent ID GUID rather than the CopilotStudio.Declarative.{title} string, so the
# legacy CASE above lands NULL. Capture the GUID only when no legacy Title ID was
# parsed, so a GUID embedded inside a declarative title is never mistaken for it.
flat = flat.withColumn('Agent_EntraId', F.expr(r'''
    CASE
      WHEN Agent_TitleID IS NOT NULL THEN NULL
      WHEN AgentId IS NULL THEN NULL
      ELSE nullif(
             regexp_extract(
               AgentId,
               '[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}',
               0),
             '')
    END
'''))


## 10. Write to Lakehouse Delta table


In [ ]:
def _validate_batch_keys(df, key_columns, dataset_name: str):
    missing = [c for c in key_columns if c not in df.columns]
    if missing:
        raise RuntimeError(
            f'{dataset_name} is missing stable key columns {missing}; cannot perform idempotent incremental merge.'
        )
    has_blank = (
        df.where(F.col('Id').isNull() | (F.trim(F.col('Id').cast('string')) == ''))
          .limit(1)
          .count()
    )
    if has_blank:
        raise RuntimeError(
            f'{dataset_name} contains blank Id values; rerun after fixing key extraction rather than appending duplicates.'
        )

def write_output(df):
    if WRITE_MODE not in ('overwrite', 'merge'):
        raise ValueError(f'Unsupported WRITE_MODE={WRITE_MODE!r}')
    if WRITE_MODE == 'merge':
        _validate_batch_keys(df, KEY_COLUMNS, 'current parsed batch')
        if spark.catalog.tableExists(OUTPUT_TABLE):
            validate_incremental_target_columns(spark.table(OUTPUT_TABLE).columns, KEY_COLUMNS, OUTPUT_TABLE)
            from delta.tables import DeltaTable
            tgt = DeltaTable.forName(spark, OUTPUT_TABLE)
            (tgt.alias('t').merge(df.alias('s'), 't.`Id` = s.`Id`')
                .whenMatchedUpdateAll()
                .whenNotMatchedInsertAll()
                .execute())
            return 'merge'
    (df.write
        .format('delta')
        .mode('overwrite')
        .option('overwriteSchema', 'true')
        .saveAsTable(OUTPUT_TABLE))
    return 'overwrite'

write_strategy = write_output(flat)
row_count = spark.table(OUTPUT_TABLE).count()
print(f'✓ Rows written to {OUTPUT_TABLE}: {row_count:,} (strategy={write_strategy})')


## 11. Verify

Spot-check the output. Expect populated `AISystemPlugin_Id` (e.g. `BingWebSearch`), `Workload`, `AppHost`.


In [ ]:
spark.table(OUTPUT_TABLE).select(
    'AISystemPlugin_Id', 'Workload', 'AppHost'
).groupBy('AISystemPlugin_Id', 'Workload', 'AppHost').count().orderBy(F.desc('count')).show(20, truncate=False)


---
**Connect the PBIT**: open the Fabric variant of either dashboard and supply the **Fabric SQL Endpoint** + **Lakehouse Database** parameters as before. The PBIT reads `Copilot_Interactions_Parsed` directly via the SQL endpoint — no other parameter changes needed when switching from the CSV-based parser to this direct ingester.
